# 01: Baseline GPT (19.3M params, 50M tokens, fixed batch size)
Standard pre-LN transformer. We first measure the largest batch that fits (for reporting), then train at a fixed **B=64** (32,768 tokens/step), a size that trains well within the 50M-token budget.

## Setup
Runtime → Change runtime type → **T4 GPU**. Results are written to Google Drive, so a disconnect does not lose finished runs.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/reversible-llm'
import os
os.makedirs(f'{DRIVE}/data', exist_ok=True)
os.makedirs(f'{DRIVE}/results', exist_ok=True)

In [ ]:
import os, sys
if not os.path.exists('/content/llm_finetune_assignments'):
    !git clone -q https://github.com/AditeyaAItronics/llm_finetune_assignments.git /content/llm_finetune_assignments
%cd /content/llm_finetune_assignments/reversible-llm-13
!git pull -q
!pip install -q tiktoken datasets
sys.path.insert(0, '/content/llm_finetune_assignments/reversible-llm-13/src')

In [ ]:
from revllm.data import prepare_tinystories
prepare_tinystories(f'{DRIVE}/data')  # tokenizes once (~55M tokens), cached on Drive
!mkdir -p /content/data && cp -n {DRIVE}/data/*.bin /content/data/

In [ ]:
import pytest
# includes the GPU memory test that is skipped on CPU
assert pytest.main(['-q', 'tests']) == 0, 'tests failed: fix before training'

In [ ]:
import json
from revllm.config import ModelConfig, TrainConfig
from revllm.batch_search import find_max_batch
from revllm.train import train

mc = ModelConfig(arch='baseline')
search = find_max_batch(mc)
json.dump(search, open(f'{DRIVE}/batch_search_baseline.json', 'w'))
search

In [ ]:
B = 64 if search['max_batch'] >= 64 else search['recommended']
res = train(mc, TrainConfig(run_name='01_baseline', batch_size=B,
                            data_dir='/content/data', out_dir=f'{DRIVE}/results'))
{k: res[k] for k in ['params', 'batch_size', 'tokens_seen', 'final_train_loss', 'final_val_loss',
                     'tokens_per_sec', 'peak_mem_allocated_gb', 'wall_time_s']}